很好，那么我们就先从这个日志开始写起吧。
实话实说，这个项目的启动是我在第一时间的想法，因为我总觉得不能再用什么LSTM去混日子，总得看点新的模型。二是在年初的核物理/标准模型的课上，我发现了这么一个ENSDF这么一个网站，其中我是觉得非常适合用于机器学习和试水diffusion model。虽然最后经过调查，我转换到了这个transformer的model上。
总得来说，这是一个增量模型，打算按照时间线去进行lora的添加，来模拟人类科学家在过去几十年的对于核物理相关参数测量的进步过程，我其实在某种意义上在期待其更多地依靠这么一个路径去预测/发现按照这么一个趋势，未来可能测出的核素和相关参数。



那么模型的导入和加载方面，我使用了pytorch相关的文件，毕竟现在它比较流行。且在整个任务施行的阶段，我大量使用了相关的生成式AI的帮助用来学习/测试代码，毕竟现在我们有了类似的工具，还是要使用的。

整个工程是需要从数据清洗-数据处理-模型输入-模型构建-回归预测，这一整套都需要我自己写，尤其是数据清洗的部分在某种意义上很让我头痛。
因为在原始文件中，不仅仅有我们需要的数据，还有一些不必要的注释和一些评级相对较低的实验数据。
在清洗的策略上，一开始想的是简单使用行内的分类，根据开头第六格的字母不同来实现简单的L G Q 的筛选，毕竟当时想的是简单做一个能跑的就行。

112PD  G 737.2     3 20      14                                             A   
112PD  L    890.3   3(0+)                                                       
（data sample）

后来在将近半个月的时间内我都在学习和调试后续模型相关，但是相对来说毫无进展，我没法训练好一个差不多还行的一个base来进行接下来的lora训练，在尝试了很多东西过后，我回头查看清洗策略的时候，才意识到似乎并不能简单使用这么一个行内关键词分类，而是得使用状态机。于是乎我又花了时间在状态机的构建上，并以它作为我们整个数据处理的核心，并会输出不只是LGQ，还有其他一切和数据相关的。

在模型的策略上，最开始的打算是一个简单的LGQ的多头输出/输入模型，先从最大的L开始，预测其中的能量值，并以此为base进行G和Q的训练。结果后来在改变到状态机了之后，还是有必要进行直接的序列相关的transformer，这个对于它来说更有优势，且更新，能更多的发挥这个模型的优点。
我的想法一开始是定了很多的输出头，最后想了想还是先聚焦在三个指标，一个分类头，一个回归头，一个用来进行我们的预测序列的长度。（好吧，就在刚刚打字的时候，我想了想还是打算把这个预测回归的长度的东西砍了好）

在这个模型优化上，是从小的overfit能走通开始，到10%-50%-100%的base，再加上lora进行后期的补充和修改。